# Deploy VSS Warehouse Blueprint 3.2

This notebook deploys the NVIDIA VSS Warehouse Blueprint 3.2 on GPU-equipped cloud instances.

In 3.2 the warehouse compose files ship **in-tree** in the `video-search-and-summarization` repo — there is no separate NGC compose tarball. On Brev, the repo is cloned onto the instance automatically; set `DEPLOY_SOURCE_PATH` in Section 1 to that path (typically `~/video-search-and-summarization`).

**What it does:**
1. Validates GPU hardware and system prerequisites (Phase 2)
2. Installs and configures the NGC CLI (Phase 1)
3. Configures Docker storage for large image pulls
4. Downloads the warehouse app data from NGC (Phase 4 — compose files come from the repo)
5. Configures `industry-profiles/warehouse-operations/.env` and deploys with Docker Compose (Phases 5 & 8)
6. Monitors and verifies the deployment (Phase 8 & 9)
7. Provides Kibana & VST access links


## 1. Configuration

Set your NGC API key and warehouse deployment parameters below.
All subsequent cells use these variables.

**Required to set yourself:**

- `NGC_CLI_API_KEY` — get from https://ngc.nvidia.com
- `DEPLOY_SOURCE_PATH` — path where the `video-search-and-summarization` repo is cloned. Default is `~/video-search-and-summarization`. The warehouse compose files now live in-tree at `<DEPLOY_SOURCE_PATH>/deploy/docker/`.

**Profile combinations:**

| MODE | BP_PROFILE | STREAM_TYPE | Dataset | Streams |
|------|-----------|-------------|---------|---------|
| `2d` | `bp_wh_kafka` | `kafka` | `warehouse-loading-dock-3cams-synthetic` | 3 |
| `2d` | `bp_wh_redis` | `redis` | `warehouse-loading-dock-3cams-synthetic` | 3 |
| `2d` | `bp_wh` | `kafka` | `nv-warehouse-4cams` | 4 |
| `3d` | `bp_wh_kafka` | `kafka` | `warehouse-4cams-20mx20m-synthetic` | 4 |
| `3d` | `bp_wh_redis` | `redis` | `warehouse-4cams-20mx20m-synthetic` | 4 |


In [ ]:
# ============================================================
# REQUIRED: Set these before running anything else
# ============================================================

NGC_CLI_API_KEY = ""          # Your NGC API key — get one at https://ngc.nvidia.com

# ---- Repo path (Brev clones the repo automatically; override if it's elsewhere) ----
import os as _os
DEPLOY_SOURCE_PATH = _os.path.expanduser("~/video-search-and-summarization")

# ---- Deployment mode ----
MODE = "2d"                   # "2d" or "3d"

# ---- Blueprint profile ----
BP_PROFILE = "bp_wh_kafka"    # "bp_wh_kafka" | "bp_wh_redis" | "bp_wh"
STREAM_TYPE = "kafka"         # "kafka" | "redis"  (auto-synced with BP_PROFILE; unused for bp_wh)

# ---- Hardware profile ----
# Valid: H100, L40, L40S, L4, A6000, RTXA6000, RTXA6000ADA, RTXPRO6000BW, IGX-THOR, DGX-SPARK
HARDWARE_PROFILE = "L40S"

# ---- Minimal vs extended (bp_wh_kafka / bp_wh_redis only) ----
# "true"  = minimal  — excludes ELK, Video Analytics API/UI, Monitoring
# ""      = extended — full deployment (includes Kibana on port 5601)
MINIMAL_PROFILE = ""

# ---- LLM / VLM (bp_wh only) ----
LLM_MODE = "none"             # "none" | "local" | "remote"
VLM_MODE = "none"             # "none" | "local" | "remote"

# ---- App-data NGC resource ----
# Default path (NVIDIAN/staging keys):
#   nvstaging/vss-warehouse/vss-warehouse-app-data:v3.2.0-05122026
# If your key resolves the production org instead, switch to nvidia/vss-warehouse/...
APP_DATA_RESOURCE = "nvstaging/vss-warehouse/vss-warehouse-app-data:v3.2.0-05122026"

# ============================================================
# OPTIONAL: Override defaults if needed
# ============================================================

# Download directory for NGC artifacts (leave empty to use home dir)
DOWNLOAD_DIR = ""             # e.g. "/home/ubuntu"

# Dataset and stream count — auto-set from MODE/BP_PROFILE if left empty / 0
SAMPLE_VIDEO_DATASET = ""     # leave empty to auto-set
NUM_STREAMS = 0               # leave 0 to auto-set

# Host IP — leave empty for auto-detect in Section 7
HOST_IP_OVERRIDE = ""


In [ ]:
# ---- Validate and resolve configuration ----
import os

assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com"
assert MODE in ("2d", "3d"), f"Invalid MODE: {MODE!r}. Must be '2d' or '3d'."
assert BP_PROFILE in ("bp_wh_kafka", "bp_wh_redis", "bp_wh"), (
    f"Invalid BP_PROFILE: {BP_PROFILE!r}. Must be 'bp_wh_kafka', 'bp_wh_redis', or 'bp_wh'."
)
if MODE == "3d" and BP_PROFILE == "bp_wh":
    raise ValueError("MODE=3d is not compatible with BP_PROFILE=bp_wh (Agents require MODE=2d).")

# Resolve and verify the repo path — compose lives in-tree under <repo>/deploy/docker
DEPLOY_SOURCE_PATH = os.path.expanduser(DEPLOY_SOURCE_PATH)
assert os.path.isdir(DEPLOY_SOURCE_PATH), (
    f"DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH!r}. "
    "On Brev the repo is cloned automatically — set DEPLOY_SOURCE_PATH in Section 1."
)
DEPLOYMENTS_DIR = os.path.join(DEPLOY_SOURCE_PATH, "deploy", "docker")
WAREHOUSE_ENV   = os.path.join(DEPLOYMENTS_DIR, "industry-profiles", "warehouse-operations", ".env")
assert os.path.isfile(os.path.join(DEPLOYMENTS_DIR, "compose.yml")), (
    f"compose.yml not found at {DEPLOYMENTS_DIR}/."
)
assert os.path.isfile(WAREHOUSE_ENV), f"Warehouse .env not found at {WAREHOUSE_ENV}."

# Sync STREAM_TYPE with BP_PROFILE
if "kafka" in BP_PROFILE:
    STREAM_TYPE = "kafka"
elif "redis" in BP_PROFILE:
    STREAM_TYPE = "redis"
else:
    STREAM_TYPE = ""  # bp_wh does not use a broker

# Auto-set dataset and stream count
_dataset = SAMPLE_VIDEO_DATASET
_streams = NUM_STREAMS
if BP_PROFILE == "bp_wh":
    _dataset = _dataset or "nv-warehouse-4cams"
    _streams = _streams or 4
    if LLM_MODE == "none": LLM_MODE = "local"
    if VLM_MODE == "none": VLM_MODE = "local"
elif MODE == "2d":
    _dataset = _dataset or "warehouse-loading-dock-3cams-synthetic"
    _streams = _streams or 3
    LLM_MODE = "none"; VLM_MODE = "none"
else:  # 3d
    _dataset = _dataset or "warehouse-4cams-20mx20m-synthetic"
    _streams = _streams or 4
    LLM_MODE = "none"; VLM_MODE = "none"
SAMPLE_VIDEO_DATASET = _dataset
NUM_STREAMS = _streams

# Export NGC key to environment for shell cells
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

# Resolve app-data extract path (NGC extracts to vss-warehouse-app-data_v<version>/vss-warehouse-app-data/)
_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")
_app_data_version = APP_DATA_RESOURCE.split(":", 1)[-1]
APP_DATA_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_data_version}", "vss-warehouse-app-data")

print("Configuration:")
print(f"  DEPLOY_SOURCE_PATH: {DEPLOY_SOURCE_PATH}")
print(f"  DEPLOYMENTS_DIR:    {DEPLOYMENTS_DIR}")
print(f"  WAREHOUSE_ENV:      {WAREHOUSE_ENV}")
print(f"  MODE:               {MODE}")
print(f"  BP_PROFILE:         {BP_PROFILE}")
print(f"  STREAM_TYPE:        {STREAM_TYPE or '(none)'}")
print(f"  HARDWARE_PROFILE:   {HARDWARE_PROFILE}")
print(f"  MINIMAL_PROFILE:    {MINIMAL_PROFILE!r}  ({'minimal' if MINIMAL_PROFILE == 'true' else 'extended'})")
print(f"  DATASET:            {SAMPLE_VIDEO_DATASET}")
print(f"  NUM_STREAMS:        {NUM_STREAMS}")
print(f"  LLM_MODE:           {LLM_MODE}")
print(f"  VLM_MODE:           {VLM_MODE}")
print(f"  APP_DATA_RESOURCE:  {APP_DATA_RESOURCE}")
print(f"  APP_DATA_DIR:       {APP_DATA_DIR}")
print(f"  NGC key:            {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")
print()
print("Configuration valid.")


## 2. Prerequisites Check

Verifies all system requirements for the Warehouse Blueprint (Phase 2):

- **2.1** NVIDIA driver and GPU detection
- **2.2** Docker 27.2.0+, non-root access, cgroupfs driver
- **2.3** NVIDIA Container Toolkit
- **2.4** Linux kernel network settings (IPv6 disabled, socket buffers)
- **2.5** IPv6 localhost entry in `/etc/hosts`
- **2.6** Minimum resources: 10+ CPU cores, 64 GB+ RAM, 500 GB+ disk

If a check fails, install/fix the missing component before proceeding.


In [ ]:
%%bash

echo "=== 2.1 GPU & NVIDIA Driver ==="
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader
echo ""

echo "=== 2.2 Docker ==="
docker --version
docker compose version
if docker ps > /dev/null 2>&1; then
    echo "Docker non-root access: OK"
else
    echo "WARNING: Docker requires sudo."
    echo "  Fix: sudo usermod -aG docker $USER && newgrp docker"
fi
echo ""

echo "=== cgroupfs driver ==="
if docker info 2>/dev/null | grep -qi "cgroupfs"; then
    echo "cgroupfs: OK"
else
    echo "WARNING: cgroupfs driver not configured in /etc/docker/daemon.json"
    echo "  Fix: add exec-opts native.cgroupdriver=cgroupfs, then restart docker"
fi
echo ""

echo "=== 2.3 NVIDIA Container Toolkit ==="
if docker run --rm --gpus all ubuntu:22.04 nvidia-smi > /dev/null 2>&1; then
    echo "NVIDIA Container Toolkit: OK"
else
    echo "WARNING: NVIDIA Container Toolkit not working."
    echo "  Install: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html"
fi
echo ""

echo "=== 2.4 Linux Kernel Settings ==="
IPV6=$(sysctl -n net.ipv6.conf.all.disable_ipv6 2>/dev/null || echo "unset")
RMEM=$(sysctl -n net.core.rmem_max 2>/dev/null || echo "0")
echo "net.ipv6.conf.all.disable_ipv6 = $IPV6 (need: 1)"
echo "net.core.rmem_max = $RMEM (need: 5242880)"
if [ "$IPV6" != "1" ]; then
    echo "WARNING: IPv6 not disabled."
fi
if [ "${RMEM:-0}" -lt 5242880 ] 2>/dev/null; then
    echo "WARNING: rmem_max too low."
fi
echo ""

echo "=== 2.5 IPv6 Localhost Entry ==="
HOSTS_ENTRY=$(grep "^::1" /etc/hosts 2>/dev/null || echo "(not found)")
echo "  /etc/hosts ::1 line: $HOSTS_ENTRY"
if echo "$HOSTS_ENTRY" | grep -q "localhost6"; then
    echo "  IPv6 localhost entry: OK"
elif echo "$HOSTS_ENTRY" | grep -q "^::1"; then
    echo "  WARNING: must use localhost6 not localhost"
    echo "    Fix: sudo sed -i 's/^::1 localhost ip6/::1 localhost6 ip6/' /etc/hosts"
else
    echo "  INFO: No ::1 entry found (OK if IPv6 is disabled)"
fi
echo ""

echo "=== 2.6 Minimum System Resources ==="
echo "CPU cores: $(nproc) (need: 10+)"
free -h | awk '/^Mem:/ {print "RAM: " $2 " (need: 64 GB+)"}'
df -h / | tail -1 | awk '{print "Disk free: " $4 " of " $2 " (need: 500 GB+)"}'
echo ""

echo "Prerequisites check complete."


## 3. Install NGC CLI

The NGC CLI is required to download the warehouse app-data. This cell installs it if not already present, then configures it with your API key.


In [ ]:
import subprocess, os, shutil

def run(cmd, **kwargs):
    """Run a shell command, raise on failure with output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# Check if NGC CLI is already installed
ngc_path = shutil.which("ngc")
if ngc_path:
    ver = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {ver}")
else:
    import platform
    arch = platform.machine()
    if arch in ("aarch64", "arm64"):
        filename = "ngccli_linux_arm64.zip"
    else:
        filename = "ngccli_linux.zip"

    NGC_CLI_VERSION = "4.13.0"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/{NGC_CLI_VERSION}/files/{filename}"

    print(f"Installing NGC CLI {NGC_CLI_VERSION} ...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")
    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(f"NGC CLI download failed — file is only {size} bytes. Check the version URL.")
    print(f"  Downloaded {size / 1024 / 1024:.1f} MB")

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    ver = run("ngc --version 2>&1 | head -1")
    print(f"  Installed: {ver}")

# Configure NGC CLI with API key — org comes from APP_DATA_RESOURCE (first path segment)
_ngc_org = APP_DATA_RESOURCE.split("/", 1)[0]
print(f"Configuring NGC CLI (org={_ngc_org})...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)
with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = {_ngc_org}
""")

print("NGC CLI configured.")
print(run("ngc config current"))


## 4. Docker & Containerd Storage

Docker images and containerd layers for VSS require **~250GB** (NIM models, DeepStream, ELK, etc.). Most GPU cloud instances ship with a small root disk (200-250GB) that **will run out of space** during deployment.

This cell auto-detects whether your root disk is too small and moves Docker and containerd storage to a larger mount. Docker **volumes** (Elasticsearch indices, uploaded videos, Kafka data) are kept on the root disk so your data persists even if the instance is stopped and the ephemeral NVMe is wiped. Images and layers are re-pulled automatically on next deploy.

**Common NVMe mount points** (auto-detected):
- AWS DLAMI: `/opt/dlami/nvme`
- Brev/Crusoe: `/ephemeral`
- Custom RAID: `/data`

To override auto-detection, set `STORAGE_ROOT` below.

In [ ]:
import subprocess, json, os, shutil

STORAGE_ROOT = ""  # Override: set to a mount path (e.g. "/mnt/data") to skip auto-detection

MIN_ROOT_FREE_GB = 350  # If root has less than this free, move storage

# --- Auto-detect large mount ---

def get_disk_free_gb(path):
    """Return free space in GB for the filesystem containing path."""
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024 ** 3)

def get_disk_total_gb(path):
    st = os.statvfs(path)
    return (st.f_blocks * st.f_frsize) / (1024 ** 3)

def find_large_mount():
    """Look for a large non-root mount suitable for Docker storage."""
    candidates = ["/opt/dlami/nvme", "/ephemeral", "/data"]
    for path in candidates:
        if os.path.isdir(path) and os.path.ismount(path):
            free = get_disk_free_gb(path)
            if free > 200:
                return path, free
    return None, 0

def find_mount_unit(mount_path):
    """Convert a mount path to a systemd mount unit name (e.g. /opt/dlami/nvme -> opt-dlami-nvme.mount)."""
    # Strip leading slash, replace remaining slashes with dashes
    unit = mount_path.strip("/").replace("/", "-") + ".mount"
    # Verify this unit exists on the system
    r = subprocess.run(["systemctl", "cat", unit], capture_output=True, text=True)
    if r.returncode == 0:
        return unit
    return None

root_free = get_disk_free_gb("/")
root_total = get_disk_total_gb("/")

print(f"Root disk: {root_free:.0f} GB free / {root_total:.0f} GB total")

if STORAGE_ROOT:
    large_mount = STORAGE_ROOT
    mount_free = get_disk_free_gb(STORAGE_ROOT)
    print(f"Using override: {STORAGE_ROOT} ({mount_free:.0f} GB free)")
    need_move = True
else:
    large_mount, mount_free = find_large_mount()
    need_move = root_free < MIN_ROOT_FREE_GB and large_mount is not None

    if large_mount:
        print(f"Large mount:    {large_mount} ({mount_free:.0f} GB free)")
    else:
        print("No large ephemeral mount detected.")

    if root_free >= MIN_ROOT_FREE_GB:
        print(f"\nRoot disk has enough space ({root_free:.0f} GB free). No storage move needed.")
    elif not large_mount:
        print(f"\nWARNING: Root disk only has {root_free:.0f} GB free and no large mount was found.")
        print("Deployment may fail due to disk space. Consider attaching a larger volume.")

if need_move:
    DOCKER_DATA_ROOT = os.path.join(large_mount, "docker")
    CONTAINERD_ROOT = os.path.join(large_mount, "containerd")
    VOLUMES_DIR = "/var/lib/docker/volumes"  # Keep volumes on persistent root disk

    print(f"\nMoving Docker and containerd storage to {large_mount}")
    print(f"  Docker images/layers: {DOCKER_DATA_ROOT}")
    print(f"  Containerd:           {CONTAINERD_ROOT}")
    print(f"  Docker volumes:       {VOLUMES_DIR} (stays on root for persistence)")

    # --- Check what needs changing ---
    daemon_json = "/etc/docker/daemon.json"
    config = {}
    try:
        with open(daemon_json) as f:
            config = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        pass

    need_daemon_json = config.get("data-root") != DOCKER_DATA_ROOT

    subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT], check=True)
    subprocess.run(["sudo", "mkdir", "-p", VOLUMES_DIR], check=True)

    volumes_link = os.path.join(DOCKER_DATA_ROOT, "volumes")
    need_volumes_symlink = not (os.path.islink(volumes_link) and os.readlink(volumes_link) == VOLUMES_DIR)

    containerd_link = "/var/lib/containerd"
    need_containerd = not (os.path.islink(containerd_link) and os.readlink(containerd_link) == CONTAINERD_ROOT)

    # Even if symlinks are correct, ensure NVMe target dirs actually exist
    # (they get wiped when ephemeral NVMe is reset on instance stop/start)
    need_target_dirs = not os.path.isdir(DOCKER_DATA_ROOT) or not os.path.isdir(CONTAINERD_ROOT)
    if need_target_dirs:
        print(f"\n  NVMe target dir(s) missing — recreating...")
        subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT, CONTAINERD_ROOT], check=True)

    if not need_daemon_json and not need_volumes_symlink and not need_containerd:
        print(f"\n  Docker data-root already set to {DOCKER_DATA_ROOT}")
        print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")
        print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

        # Always ensure the boot-time restore service is up to date
        # (handles the case where service exists but is missing mount dependencies)
        _update_restore_service = True
        _need_restart = need_target_dirs  # Restart Docker/containerd if we had to recreate dirs
    else:
        _update_restore_service = True
        _need_restart = True

        # Stop Docker AND docker.socket (socket can reactivate Docker and recreate dirs)
        print("\n  Stopping Docker and containerd for storage reconfiguration...")
        subprocess.run(["sudo", "systemctl", "stop", "docker.socket"], check=False)
        subprocess.run(["sudo", "systemctl", "stop", "docker"], check=True)
        subprocess.run(["sudo", "systemctl", "stop", "containerd"], check=True)

        # --- Docker daemon.json ---
        if need_daemon_json:
            config["data-root"] = DOCKER_DATA_ROOT
            new_config = json.dumps(config, indent=2)
            subprocess.run(
                f"echo '{new_config}' | sudo tee {daemon_json}",
                shell=True, check=True, capture_output=True
            )
            print(f"  Docker data-root set to {DOCKER_DATA_ROOT}")
        else:
            print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")

        # --- Volumes symlink (use ln -sfn for idempotency) ---
        if need_volumes_symlink:
            # ln -sfn: force, no-dereference (replaces existing dir/symlink atomically)
            subprocess.run(["sudo", "rm", "-rf", volumes_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", VOLUMES_DIR, volumes_link], check=True)
            print(f"  Created symlink: {volumes_link} -> {VOLUMES_DIR}")
        else:
            print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")

        # --- Containerd ---
        if need_containerd:
            subprocess.run(["sudo", "mkdir", "-p", CONTAINERD_ROOT], check=True)
            if os.path.isdir(containerd_link) and not os.path.islink(containerd_link):
                # Move existing containerd data
                subprocess.run(f"sudo mv {containerd_link}/* {CONTAINERD_ROOT}/ 2>/dev/null; true",
                               shell=True, check=False)
                subprocess.run(["sudo", "rm", "-rf", containerd_link], check=True)
                print(f"  Containerd data moved to {CONTAINERD_ROOT}")
            elif os.path.lexists(containerd_link):
                subprocess.run(["sudo", "rm", "-f", containerd_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", CONTAINERD_ROOT, containerd_link], check=True)
            print(f"  Containerd symlinked: {containerd_link} -> {CONTAINERD_ROOT}")
        else:
            print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

    # --- Install/update boot-time restore service ---
    # Ephemeral NVMe is wiped on instance stop/start. This systemd service
    # recreates the directories before Docker/containerd start so they don't crash-loop.
    # We use RequiresMountsFor= so the service waits for the NVMe to actually be mounted.
    if _update_restore_service:
        unit_name = "docker-nvme-restore.service"
        unit_path = f"/etc/systemd/system/{unit_name}"

        # Build After= line — include the mount unit if systemd knows about it
        after_targets = "local-fs.target"
        mount_unit = find_mount_unit(large_mount)
        if mount_unit:
            after_targets += f" {mount_unit}"

        unit_content = f"""[Unit]
Description=Restore Docker/containerd dirs on ephemeral NVMe
Before=containerd.service docker.service
After={after_targets}
RequiresMountsFor={large_mount}

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'mkdir -p {DOCKER_DATA_ROOT} {CONTAINERD_ROOT}'

[Install]
WantedBy=multi-user.target
"""
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.service', delete=False) as tmp:
            tmp.write(unit_content)
            tmp_path = tmp.name
        subprocess.run(["sudo", "cp", tmp_path, unit_path], check=True)
        os.unlink(tmp_path)
        subprocess.run(["sudo", "systemctl", "daemon-reload"], check=True)
        subprocess.run(["sudo", "systemctl", "enable", unit_name], check=True, capture_output=True)
        print(f"  Installed {unit_name} (restores NVMe dirs on boot, waits for mount)")

    # --- Restart if needed ---
    if _need_restart:
        print("\n  Starting containerd and Docker...")
        subprocess.run(["sudo", "systemctl", "start", "containerd"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker.socket"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)

    r = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
    print(f"\n  Docker data-root: {r.stdout.strip()}")
    target = os.readlink(containerd_link) if os.path.islink(containerd_link) else containerd_link
    print(f"  Containerd root:  {target}")
    print(f"\n  Storage configuration complete.")
else:
    if not STORAGE_ROOT and root_free >= MIN_ROOT_FREE_GB:
        print("Skipping storage move.")

## 5. Docker Login

Authenticate with the NVIDIA Container Registry (`nvcr.io`) to pull deployment images.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "login", "nvcr.io",
     "--username", "$oauthtoken",
     "--password", NGC_CLI_API_KEY],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Docker login to nvcr.io: OK")
else:
    print(f"Docker login FAILED:\n{result.stderr}")
    raise RuntimeError("Docker login to nvcr.io failed")

## 6. Get Deployment Code

**Compose bundle (3.2 change):** the warehouse compose files ship **in-tree** in the `video-search-and-summarization` repo — there is no separate compose tarball to download. On Brev, the repo is cloned automatically; Section 1 verified `DEPLOY_SOURCE_PATH` points at it.

**App data** (videos + pre-trained models) still comes from NGC:

- Default: `nvstaging/vss-warehouse/vss-warehouse-app-data:v3.2.0-05122026`
- If your key resolves the production org instead, switch `APP_DATA_RESOURCE` in Section 1 to `nvidia/vss-warehouse/vss-warehouse-app-data:<version>`.

> **First run only** — if the app-data directory already exists on disk, the download is skipped.


In [ ]:
import subprocess, os

_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# ---- Compose bundle: in-tree, no download ----
print(f"Compose bundle (in-tree): {DEPLOYMENTS_DIR}")
print(f"  compose.yml:    {os.path.join(DEPLOYMENTS_DIR, 'compose.yml')}")
print(f"  warehouse .env: {WAREHOUSE_ENV}")
print()

# ---- App data ----
_app_data_version = APP_DATA_RESOURCE.split(":", 1)[-1]
APP_DATA_EXTRACT_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_data_version}")
APP_DATA_DIR         = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data")

if os.path.isdir(APP_DATA_DIR):
    print(f"App data already present: {APP_DATA_DIR}")
else:
    print(f"Downloading app data ({APP_DATA_RESOURCE}) ...")
    run(f"ngc registry resource download-version {APP_DATA_RESOURCE!r}", cwd=_base)
    tarball = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data.tar.gz")
    print(f"Extracting {os.path.basename(tarball)} ...")
    run(f"tar -xf {tarball!r}", cwd=APP_DATA_EXTRACT_DIR)
    print("App data extracted.")

run(f"sudo chmod -R 777 {APP_DATA_DIR!r}")
print(f"APP_DATA_DIR:    {APP_DATA_DIR}")
print()
print("Deployment artifacts ready.")


## 7. Detect Network Configuration

Auto-detects the internal IP (`HOST_IP`) used in `warehouse/.env` for inter-container communication.
Set `HOST_IP_OVERRIDE` in Section 1 if auto-detection gives the wrong result.

Also reads `BREV_ENV_ID` for generating the Kibana secure link in Section 10.


In [ ]:
import subprocess, os

def detect_internal_ip():
    out = subprocess.run(
        ["ip", "-4", "route", "get", "1.1.1.1"],
        capture_output=True, text=True, timeout=5,
    )
    parts = out.stdout.split()
    for i, p in enumerate(parts):
        if p == "src" and i + 1 < len(parts):
          return parts[i + 1]
    return ""

def read_etc_environment():
    env = {}
    try:
        with open("/etc/environment") as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    key, _, value = line.partition("=")
                    env[key.strip()] = value.strip().strip('"')
    except FileNotFoundError:
        pass
    return env

HOST_IP = HOST_IP_OVERRIDE or detect_internal_ip()
print(f"HOST_IP: {HOST_IP}")
if not HOST_IP:
    print("WARNING: Could not detect HOST_IP. Set HOST_IP_OVERRIDE in Section 1.")

# --- Brev Secure Links ---
# On Brev, all browser-facing traffic routes through an nginx reverse proxy
# on a single port (default 7777). This avoids CORS issues with Cloudflare
# Access when each port gets its own hostname.
# Check os.environ first, then fall back to /etc/environment (Jupyter kernels
# may not inherit /etc/environment depending on how the notebook server starts).
_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env.get("BREV_ENV_ID", "")
if BREV_ENV_ID:
    # Ensure it's in os.environ so dev-profile.sh picks it up
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    print(f"\n=== Brev Environment Detected ===")
    print(f"  BREV_ENV_ID: {BREV_ENV_ID}")
else:
    BREV_ENV_ID = ""  # ensure defined for later cells
    print("No Brev environment detected")


## 8. Deploy

This cell (Phases 5 & 8):
1. Writes the warehouse `.env` at `<repo>/deploy/docker/industry-profiles/warehouse-operations/.env` with your settings from Section 1 (template defaults are preserved for everything else)
2. Logs in to `nvcr.io`
3. Runs `docker compose up` from `<repo>/deploy/docker/`:

   ```bash
   docker compose -f compose.yml \
     --env-file industry-profiles/warehouse-operations/.env \
     up --detach --pull always --force-recreate --build
   ```

**Expected time: 10–20 minutes** on first run (image pulls + perception engine build).
Subsequent runs are faster — images stay in the local cache and the TensorRT engine is cached under `$VSS_DATA_DIR/models/`.

Full log is written to `~/deploy_warehouse.log`. Monitor progress in Section 9.


In [ ]:
import os, re, shlex, subprocess, time

LOG_FILE = os.path.expanduser("~/deploy_warehouse.log")
ENV_FILE = WAREHOUSE_ENV  # <repo>/deploy/docker/industry-profiles/warehouse-operations/.env

def merge_warehouse_dotenv(path, assignments):
    """Update KEY=value lines in the warehouse .env in place; preserve all other variables.

    Returns the set of keys written by this merge (replaced or appended).
    """
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    used = set()
    out = []
    for line in lines:
        s = line.strip()
        if not s or s.startswith("#"):
            out.append(line)
            continue
        if "=" not in s:
            out.append(line)
            continue
        key, _, _ = s.partition("=")
        key = key.strip()
        if key in assignments:
            val = assignments[key]
            out.append(f"{key}={shlex.quote(val)}\n")
            used.add(key)
        else:
            out.append(line)
    for k in assignments:
        if k not in used:
            out.append(f"{k}={shlex.quote(assignments[k])}\n")
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(out)
    return set(assignments)

# ---- Phase 5: Merge into <repo>/deploy/docker/industry-profiles/warehouse-operations/.env ----
print(f"Merging notebook settings into {ENV_FILE} ...")
assert os.path.isfile(ENV_FILE), f"Missing {ENV_FILE}. Re-run Section 6."

_env_updates = {
    "MODE": MODE,
    "BP_PROFILE": BP_PROFILE,
    "STREAM_TYPE": STREAM_TYPE,
    "MINIMAL_PROFILE": MINIMAL_PROFILE,
    "SAMPLE_VIDEO_DATASET": SAMPLE_VIDEO_DATASET,
    "NUM_STREAMS": str(NUM_STREAMS),
    "HARDWARE_PROFILE": HARDWARE_PROFILE,
    "LLM_MODE": LLM_MODE,
    "VLM_MODE": VLM_MODE,
    "VSS_APPS_DIR": DEPLOYMENTS_DIR,
    "VSS_DATA_DIR": APP_DATA_DIR,
    "HOST_IP": HOST_IP,
    "NGC_CLI_API_KEY": NGC_CLI_API_KEY,
}
_written = merge_warehouse_dotenv(ENV_FILE, _env_updates)
_preview = {k: (f"{v[:4]}...{v[-4:]}" if k == "NGC_CLI_API_KEY" and len(v) > 12 else v) for k, v in _env_updates.items()}
print(f"warehouse .env updated (template defaults preserved). Keys merged: {sorted(_written)}")
print(f"Values (NGC_CLI_API_KEY redacted): {_preview}")
print()

# ---- Phase 8: Bring up ----
print("Logging in to nvcr.io ...")
r = subprocess.run(
    ["docker", "login", "--username", "$oauthtoken", "--password", NGC_CLI_API_KEY, "nvcr.io"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(f"Docker login failed:\n{r.stderr}")
print("Docker login: OK")
print()

# Run docker compose from <repo>/deploy/docker.
_cmd = [
    "docker", "compose",
    "--progress", "plain",
    "-f", "compose.yml",
    "--env-file", "industry-profiles/warehouse-operations/.env",
    "up",
    "--detach",
    "--pull", "always",
    "--force-recreate",
    "--build",
]
_compose_env = {**os.environ, "NGC_CLI_API_KEY": NGC_CLI_API_KEY, "NO_COLOR": "1"}

# --- Parse docker compose plain output ---
PULLING_RE = re.compile(r"(?:^|\s)(\S+)\s+Pulling\b|Pulling\s+(?:from\s+)?(\S+)")
PULLED_RE = re.compile(r"(?:✔\s*)?(\S+)\s+Pulled\b")
BUILD_STEP_RE = re.compile(r"#\d+\s+\[([^\]]+)\]\s+(.+)")
IMAGE_BUILT_RE = re.compile(r"(?:✔\s*)?(\S+)\s+Built\b|Successfully\s+built\s+(\S+)")
CONTAINER_RE = re.compile(r"^\s*Container\s+(\S+)\s+(.+?)\s*$")

SUPPRESS_PATTERNS = [re.compile(r"^\s*$"), re.compile(r"^#[0-9]+\s+\[internal\]")]
PHASE_PATTERNS = [
    (re.compile(r"\[\+\]\s*Pulling", re.I), "pulling"),
    (re.compile(r"\[\+\]\s*Building", re.I), "building"),
    (re.compile(r"Creating network|Network\s+\S+\s+Created", re.I), "network"),
    (re.compile(r"Creating\s|Created\s+container", re.I), "containers"),
]

errors = []
images_pulling = set()
images_pulled = set()
builds = {}
builds_done = set()
containers = {}
phases_seen = []
STATUS_REFRESH_INTERVAL_S = 10
last_refresh = 0.0
_t0 = time.monotonic()


def _elapsed():
    return time.monotonic() - _t0


def print_status():
    print("=" * 50, flush=True)
    print(f"Elapsed {_elapsed():.0f}s  |  errors: {len(errors)}", flush=True)
    print(
        f"  pull: {len(images_pulling)} in flight, {len(images_pulled)} done  |  "
        f"build services: {len(builds)} active, {len(builds_done)} done  |  "
        f"containers: {len(containers)}",
        flush=True,
    )
    if phases_seen:
        print("  phases:", " → ".join(phases_seen[-8:]), flush=True)
    if builds:
        tail = list(builds.items())[-3:]
        print("  build:", "; ".join(f"{k}: {v[:48]}" for k, v in tail), flush=True)
    if containers:
        tail = list(containers.items())[-5:]
        print("  last containers:", "; ".join(f"{n}={s}" for n, s in tail), flush=True)
    if errors:
        print(f"  last error line: {errors[-1][:200]}", flush=True)
    print("=" * 50, flush=True)


print(f"Command: {' '.join(_cmd)}")
print(f"Working dir: {DEPLOYMENTS_DIR}")
print(f"Logging to {LOG_FILE} — status keeps updating when something changes.")
print()

process = subprocess.Popen(
    _cmd,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=DEPLOYMENTS_DIR,
    env=_compose_env,
)
assert process.stdout is not None

rc = None
try:
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.rstrip()

            if "[ERROR]" in stripped:
                errors.append(stripped)
                print_status()
                continue

            m_pulling = PULLING_RE.search(stripped)
            if m_pulling:
                name = next((g for g in m_pulling.groups() if g), None)
                if name:
                    images_pulling.add(name)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_pulled = PULLED_RE.search(stripped)
            if m_pulled:
                name = m_pulled.group(1)
                images_pulled.add(name)
                images_pulling.discard(name)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_build = BUILD_STEP_RE.match(stripped)
            if m_build:
                svc, step = m_build.group(1), m_build.group(2)
                builds[svc] = step
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_built = IMAGE_BUILT_RE.search(stripped)
            if m_built:
                svc = next((g for g in m_built.groups() if g), None)
                if svc:
                    builds_done.add(svc)
                    builds.pop(svc, None)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            if any(p.search(stripped) for p in SUPPRESS_PATTERNS):
                continue

            for pattern, label in PHASE_PATTERNS:
                if pattern.search(stripped):
                    if label not in phases_seen:
                        phases_seen.append(label)
                        print_status()
                    break

            m = CONTAINER_RE.match(stripped)
            if m:
                name, status = m.group(1), m.group(2)
                if status.startswith("Exited"):
                    status = "Exited"
                containers[name] = status
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()

        rc = process.wait()
        log.write(f"\n---- docker compose finished (exit code {rc}) ----\n")
        log.flush()

    print_status()
    print("=" * 50)
    if rc == 0 and not errors:
        print(f"Deployment complete in {_elapsed():.1f}s.")
    else:
        print(f"\nDeployment FAILED (exit code {rc}).")
        if errors:
            print(f"\n{len(errors)} error line(s) captured — see above and log.")
        print(f"\nFull log: {LOG_FILE}")
        print(f"  View with: cat {LOG_FILE}")

except KeyboardInterrupt:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=120)
        except Exception:
            pass
    raise
finally:
    if process.stdout is not None:
        try:
            process.stdout.close()
        except Exception:
            pass

if rc is None:
    raise RuntimeError(f"docker compose did not complete normally. See {LOG_FILE}.")
if rc != 0:
    raise RuntimeError(f"docker compose exited with {rc}. See {LOG_FILE} for full output.")
print(f"\nApp data dir: {APP_DATA_DIR}")
print("Run Section 9 to verify containers / FPS.")


## 9. Verify Deployment

Polls container status and checks FPS output from the perception container.
Re-run this cell every minute until all core containers show `Up`.

**Stack is ready when these containers show `Up`** (3.2 uses the same container names in 2D and 3D — no `-2d` / `-3d` suffix):

- `vss-vios-nvstreamer`, `vss-rtvi-cv`, `vss-rtvi-cv-sdr`, `vss-configurator`, `vss-behavior-analytics`
- VST stack: `vss-vios-postgres`, `-sensor`, `-streamprocessing`, `-sdr`, `-ingress`, `-envoy`
- Auto-calibration: `vss-auto-calibration`, `vss-auto-calibration-ui`
- Broker: `kafka` or `redis`, plus `vss-broker-health-check`
- Extended profile adds: `elasticsearch`, `kibana`, `logstash`, `vss-video-analytics-api`, `vss-haproxy-ingress`

> **First-run perception is slow:** `vss-rtvi-cv` compiles the TensorRT engine on its first start. The engine is cached under `$VSS_DATA_DIR/models/mtmc/` so subsequent starts skip the compile. FPS lines only appear once the engine load finishes.


In [ ]:
import subprocess, os

# ---- Container status ----
print("=== Running Containers ===")
subprocess.run(["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"])
print()

# ---- FPS check ----
# In 3.2 the perception container is `vss-rtvi-cv` in both 2D and 3D.
container = "vss-rtvi-cv"
print(f"=== FPS check ({container}) ===")
r = subprocess.run(
    ["docker", "logs", "--tail", "50", container],
    capture_output=True, text=True,
)
output = (r.stdout or "") + (r.stderr or "")
if r.returncode != 0:
    if output.strip():
        print(f"docker logs exit {r.returncode} for {container}:\n{output[-800:]}")
    else:
        print(f"docker logs failed for {container} (exit {r.returncode}). Is the stack up?")
else:
    # Perception logs report throughput as DeepStream PERF lines (no literal "fps").
    def _is_throughput_log_line(line: str) -> bool:
        lo = line.lower()
        return (
            "fps" in lo
            or "perf" in lo
            or "active sources" in lo
            or ("source_id" in lo and "stream_name" in lo)
        )

    perf_lines = [l for l in output.splitlines() if _is_throughput_log_line(l)]
    if perf_lines:
        for line in perf_lines[-8:]:
            print(line)
    else:
        print(f"No FPS / PERF output yet from {container}.")
        print("Re-run this cell in ~1 minute once the stack is fully up.")
        print()
        print(f"docker logs --tail 10 {container}:")
        r_tail = subprocess.run(
            ["docker", "logs", "--tail", "10", container],
            capture_output=True, text=True,
        )
        tail_out = (r_tail.stdout or "") + (r_tail.stderr or "")
        if tail_out.strip():
            print(tail_out.rstrip())
        else:
            print("(no log lines)")


## 10. Access Kibana & VST (Video Stream Tool UI)

In 3.2, Kibana and VST are both reached through the **HAProxy ingress on port 7777**, with path-based routing:

- **Kibana** → `http://<host>:7777/kibana/` (requires extended profile: `MINIMAL_PROFILE=""`)
- **VST UI** → `http://<host>:7777/vst/`

You only need **one** open/forwarded port (`7777`) instead of separate ports for each service.

### Streaming limitations (Live / Recorded)

The VST live and recorded streams use **WebRTC media over UDP (RTP)**, which is **not** carried by a tunneled TCP port. A **TCP** port-forward on 7777 (e.g. `brev port-forward`) still does not forward that UDP path.

Run the cell below for links.


In [ ]:
import os

INGRESS_PORT = 7777

if BREV_ENV_ID:
    # Brev secure link format: "<port>-<env_id>.brevlab.com"
    base = f"https://{INGRESS_PORT}-{BREV_ENV_ID}.brevlab.com"
    kibana_url = f"{base}/kibana/"
    vst_url    = f"{base}/vst/"
    print(f"Kibana (via Brev secure link): {kibana_url}")
    print()
    print("Setup:")
    print(f"  1. In the Brev dashboard, create a secure link for port {INGRESS_PORT}")
    print(f"  2. Open Kibana: {kibana_url}")
    print(f"  3. Open VST UI: {vst_url}")
    print()
    print("Note: Live/recorded stream playback uses WebRTC UDP; tunnel TCP alone may not be enough.")
else:
    base = f"http://{HOST_IP}:{INGRESS_PORT}"
    kibana_url = f"{base}/kibana/"
    vst_url    = f"{base}/vst/"
    print(f"Kibana: {kibana_url}")
    print(f"VST UI: {vst_url}")
    print()
    print(f"If port {INGRESS_PORT} isn't directly accessible, use SSH port forwarding:")
    print(f"  ssh -L {INGRESS_PORT}:localhost:{INGRESS_PORT} <user>@{HOST_IP}")
    print(f"  Then open: http://localhost:{INGRESS_PORT}/kibana/  and  http://localhost:{INGRESS_PORT}/vst/")
    print()
    print("Note: Live/recorded stream playback uses WebRTC UDP; open the UDP port range your deployment needs on the firewall.")


## 11. Next Steps

Once Kibana is accessible, open the **Discover** or **Dashboard** tab to view behavior analytics events
(ROI crossings, proximity alerts, object tracking metadata) flowing from the warehouse deployment.

For more details see the [VSS Warehouse Blueprint documentation](https://docs.nvidia.com/vss/3.2.0/).


## 12. Stop Deployment

Stops all containers **without deleting data or volumes**.
Re-run Section 8 to restart the stack.


In [ ]:
import subprocess, os

print("Stopping warehouse stack (env: industry-profiles/warehouse-operations/.env) ...")
r = subprocess.run(
    ["docker", "compose",
     "-f", "compose.yml",
     "--env-file", "industry-profiles/warehouse-operations/.env",
     "stop"],
    cwd=DEPLOYMENTS_DIR, capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("docker compose stop failed")
print("Stack stopped.")


## 13. Teardown

Stops and removes all containers, networks, and volumes. Data in `$VSS_DATA_DIR` (app data, including the cached TensorRT engine) is preserved on disk. Re-run Sections 8–9 to redeploy.


In [ ]:
import subprocess, os

ENV_REL = "industry-profiles/warehouse-operations/.env"

print("Tearing down warehouse stack ...")

for cmd in [
    ["docker", "compose", "-f", "compose.yml", "--env-file", ENV_REL, "down"],
    ["docker", "volume", "prune", "-f"],
    ["docker", "system", "prune", "-f"],
]:
    print(f"  {' '.join(cmd)}")
    r = subprocess.run(cmd, cwd=DEPLOYMENTS_DIR, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.returncode != 0:
        print(f"  WARNING: {r.stderr}")

# Cleanup datalog scripts — in 3.2 the script lives in-tree.
cleanup = os.path.join(DEPLOYMENTS_DIR, "scripts", "cleanup_all_datalog.sh")
if os.path.isfile(cleanup):
    print(f"\n  bash {cleanup} -e {ENV_REL}")
    r = subprocess.run(
        ["bash", cleanup, "-e", ENV_REL],
        cwd=DEPLOYMENTS_DIR, capture_output=True, text=True,
    )
    if r.stdout.strip():
        print(r.stdout)

print("Teardown complete.")


In [ ]:
# To remove the downloaded app-data tarball + extracted dir from disk:
# (the repo at DEPLOY_SOURCE_PATH is NOT removed — that's a git checkout, not a notebook artifact)
#
# import shutil
# shutil.rmtree(APP_DATA_EXTRACT_DIR)   # <home>/vss-warehouse-app-data_v<version>/
# print("App-data artifacts removed.")
